In [162]:
import sys
from pathlib import Path
from pyprojroot import here

sys.path.append(str(here()))


In [163]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [164]:
from src.config import cfg
from src.my_utils import set_seed

In [165]:
gseed = cfg.general.seed
set_seed(gseed)

In [166]:
train_path = Path(cfg.paths.train)
submit_path = Path(cfg.paths.test)

In [167]:
df_train = pd.read_csv(train_path)
df_submit = pd.read_csv(submit_path)

### микро план
* Минимальная предобработка (заполнить пропуски) и обработать категории
* 1) Масштабируем + logreg
  2) random forest
  3) lightGBM
* сравнить метрики



### Nans

In [168]:
# train
miss = df_train.isna().sum()
miss = miss[miss > 0]

pd.DataFrame({"mean": miss / df_train.shape[0], "sum": miss})

,mean,sum
Age,0.198653,177
Cabin,0.771044,687
Embarked,0.002245,2


In [169]:
# test
miss = df_submit.isna().sum()
miss = miss[miss > 0]

pd.DataFrame({"mean": miss / df_submit.shape[0], "sum": miss})

,mean,sum
Age,0.205742,86
Fare,0.002392,1
Cabin,0.782297,327


In [170]:
df_train.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [171]:
X = df_train.drop(columns=["Survived"])
y = df_train["Survived"]

In [172]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [173]:
cat_columns = ["Pclass", "Sex", "Embarked"]
num_columns = ["Age", "SibSp", "Parch", "Fare"]

In [ ]:
# логистическая регрессия
cat_transformer_lin = Pipeline(
    [
        ("cat_imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(drop="first")),
    ]
)

num_transformer_lin = Pipeline(
    [("num_imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

preprocessor_lin = ColumnTransformer(
    [
        ("cat", cat_transformer_lin, cat_columns),
        ("num", num_transformer_lin, num_columns),
    ]
)

In [175]:
# деревянные
cat_transformer_tree = Pipeline(
    [
        ("cat_imputer", SimpleImputer(strategy="most_frequent")),
        # (
        #     "ordinal_encoder",
        #     OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        # ),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]
)

num_transformer_tree = Pipeline([("num_imputer", SimpleImputer(strategy="median"))])

preprocessor_tree = ColumnTransformer(
    [
        ("cat", cat_transformer_tree, cat_columns),
        ("num", num_transformer_tree, num_columns),
    ]
)

Т.к люди из одной семьи встречаются и в train, и в test, то лучше сделать разбиение и валидацию со стратификацией, намеренное выделение в группы по семье и билету будет лишним усложнением


In [176]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [177]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=gseed)

In [178]:
df_train["Pclass"] = df_train["Pclass"].astype(str)
df_submit["Pclass"] = df_submit["Pclass"].astype(str)


In [179]:
def baseline_report(model, X, y, cv_splitter, preprocessor):
    model_pipe = Pipeline(
        [
            ("preprocessing", preprocessor),
            ("model", model),
        ]
    )
    cv_results = cross_validate(
        model_pipe,
        X,
        y,
        cv=cv_splitter,
        scoring="accuracy",
        return_train_score=True,
        return_estimator=True,
    )

    fitted_models = cv_results["estimator"]

    metrics = pd.DataFrame(cv_results)[["train_score", "test_score"]].agg(
        ["mean", "std"]
    )

    return fitted_models, metrics

In [180]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

In [191]:
# логистическая регрессия
logreg_models, metrics = baseline_report(
    LogisticRegression(), X, y, skf, preprocessor_lin
)
metrics

,train_score,test_score
mean,0.806585,0.801373
std,0.005504,0.049563


In [182]:
# случайный лес 1
random_forest_models, metrics = baseline_report(
    RandomForestClassifier(random_state=gseed), X, y, skf, preprocessor_tree
)
metrics  # явное переобучение

,train_score,test_score
mean,0.981419,0.811436
std,0.002073,0.049957


In [183]:
# случайный лес 2
random_forest_models, metrics = baseline_report(
    RandomForestClassifier(random_state=gseed, max_depth=7),
    X,
    y,
    skf,
    preprocessor_tree,
)
metrics

,train_score,test_score
mean,0.890510,0.826042
std,0.004993,0.037891


In [184]:
# градиентный бустинг 1
lgbm_models, metrics = baseline_report(
    LGBMClassifier(random_state=gseed), X, y, skf, preprocessor_tree
)
metrics  # переобучение

[LightGBM] [Info] Number of positive: 307, number of negative: 494
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000683 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 221
[LightGBM] [Info] Number of data points in the train set: 801, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383271 -> initscore=-0.475688
[LightGBM] [Info] Start training from score -0.475688
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

,train_score,test_score
mean,0.948123,0.831623
std,0.004707,0.029632


In [185]:
# градиентный бустинг 2
lgbm_models, metrics = baseline_report(
    LGBMClassifier(random_state=gseed, max_depth=7), X, y, skf, preprocessor_tree
)
metrics

[LightGBM] [Info] Number of positive: 307, number of negative: 494
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000274 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 221
[LightGBM] [Info] Number of data points in the train set: 801, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383271 -> initscore=-0.475688
[LightGBM] [Info] Start training from score -0.475688
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

,train_score,test_score
mean,0.916075,0.838352
std,0.005636,0.028290


На кросс валидации бустинг показал лучшую точность, теперь это baseline

In [186]:
baseline_model = LGBMClassifier(random_state=gseed, max_depth=7)

In [187]:
baseline_pipe = Pipeline(
    [
        ("preprocessing", preprocessor_tree),
        ("model", baseline_model),
    ]
)

In [188]:
baseline_pipe.fit(X, y)
preds = baseline_pipe.predict_proba(df_submit)[:, 1]
submit_preds = (preds >= 0.5).astype(int)

[LightGBM] [Info] Number of positive: 342, number of negative: 549
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000188 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 228
[LightGBM] [Info] Number of data points in the train set: 891, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383838 -> initscore=-0.473288
[LightGBM] [Info] Start training from score -0.473288
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [189]:
submission = pd.DataFrame(
    {"PassengerId": df_submit["PassengerId"], "Survived": submit_preds}
)
submission.to_csv(Path(cfg.paths.output) / "baseline_submission.csv", index=False)

### Итог:
#### Ordinal Encoding
* Модель: LightGBM
* Скор на валидации: 0.837228
* Скор сабмита: 0.72966

#### One Hot Encoding
* Модель: LightGBM
* Скор на валидации: 0.838352
* Скор сабмита: 0.77033